# Importação das bibliotecas necessárias

In [1]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

.::Métodos disponíveis::.
------------------------------------------------------
extrair_dados_bronze(self,arquivo='')
encontrar_ano_mes(caminho_prefixo)
criar_view_temporaria_arquivo(local, nome, schema=None, aba=None, **config)
atualizar_tabela_delta(arquivo)
criar_view_temporaria(df, nome)
historico_tabela()
carrega_lista_yaml()
salvar_delta(df, mode)
parar_sessao()



# Crie uma instância da classe AlinharETL

In [2]:
# Crie uma instância da classe
bucket = "silver"
datamart = "compliance"
extrator_bronze = AlinharETL(bucket,datamart)

2024-09-26 19:29:16,104 - INFO - Iniciando Sessão Spark.


# Leitura das tabelas

In [3]:
df_compliance_cbo_sub_grupo = extrator_bronze.criar_view_temporaria('silver/Mte/MteCBOSubGrupo','compliance_cbo_sub_grupo')

2024-09-26 19:29:20,490 - INFO - Aguarde... Criando View (silver/Mte/MteCBOSubGrupo)
2024-09-26 19:29:27,067 - INFO - View criada com sucesso


In [4]:
df_compliance_cbo_grande_sub_grupo_principal = extrator_bronze.criar_view_temporaria('silver/Mte/MteCBOSubGrupoPrincipal','compliance_flat_sub_grupo_principal')

2024-09-26 19:29:27,072 - INFO - Aguarde... Criando View (silver/Mte/MteCBOSubGrupoPrincipal)
2024-09-26 19:29:27,316 - INFO - View criada com sucesso


In [5]:
df_compliance_cbo_familia= extrator_bronze.criar_view_temporaria('silver/Mte/MteCBOFamilia','compliance_cbo_familia')

2024-09-26 19:29:27,320 - INFO - Aguarde... Criando View (silver/Mte/MteCBOFamilia)
2024-09-26 19:29:27,518 - INFO - View criada com sucesso


In [6]:
df_compliance_cbo_grande_grupo = extrator_bronze.criar_view_temporaria('silver/Mte/MteCBOGrandeGrupo','compliance_cbo_grande_grupo')

2024-09-26 19:29:27,525 - INFO - Aguarde... Criando View (silver/Mte/MteCBOGrandeGrupo)
2024-09-26 19:29:27,747 - INFO - View criada com sucesso


In [7]:
df_compliance_cbo_ocupacao = extrator_bronze.criar_view_temporaria('silver/Mte/MteCBOOcupacao','compliance_cbo_ocupacao')

2024-09-26 19:29:27,751 - INFO - Aguarde... Criando View (silver/Mte/MteCBOOcupacao)
2024-09-26 19:29:27,923 - INFO - View criada com sucesso


# Tratamentos na tabela 

In [8]:
sql_query = """
SELECT 
    s.cd AS cd,
    p.demanda_formacao,
    p.grande_grupo AS dsc_grande_grupo,
    p.subgrupo_principal AS dsc_subgrupo_principal,
    s.TITULO AS dsc_subgrupo,
    p.familia AS dsc_formacao,
    p.ocupacao AS dsc_ocupacao,
    'SUBGRUPO' AS dsc_nivel,
    p.industrial_uniepro AS dsc_industrial_uniepro
FROM 
    compliance_cbo_sub_grupo s
LEFT OUTER JOIN 
    compliance_flat_sub_grupo_principal p ON s.cd_subgrupo_principal = p.codigo

UNION ALL

SELECT 
    f.CODIGO AS cd,
    s.demanda_formacao,
    s.grande_grupo AS dsc_grande_grupo,
    s.subgrupo_principal AS dsc_subgrupo_principal,
    s.subgrupo AS dsc_subgrupo,
    f.TITULO AS dsc_formacao,
    s.ocupacao AS dsc_ocupacao,
    'FAMILIA' AS dsc_nivel,
    s.industrial_uniepro AS dsc_industrial_uniepro
FROM 
    compliance_cbo_familia f
LEFT JOIN 
    (SELECT 
        s.cd AS codigo,
        p.demanda_formacao,
        p.grande_grupo,
        p.subgrupo_principal,
        s.TITULO AS subgrupo,
        p.familia,
        p.ocupacao,
        p.industrial_uniepro
     FROM 
        compliance_cbo_sub_grupo s
     LEFT OUTER JOIN 
        compliance_flat_sub_grupo_principal p ON s.cd_subgrupo_principal = p.codigo) AS s
ON 
    f.cd_subgrupo = s.codigo

UNION ALL

SELECT 
    o.CODIGO AS cd,
    f.demanda_formacao,
    f.grande_grupo AS dsc_grande_grupo,
    f.subgrupo_principal AS dsc_subgrupo_principal,
    f.subgrupo AS dsc_subgrupo,
    f.familia AS dsc_formacao,
    o.TITULO AS dsc_ocupacao,
    'OCUPACAO' AS dsc_nivel,
    f.industrial_uniepro AS dsc_industrial_uniepro
FROM 
    compliance_cbo_ocupacao o
LEFT JOIN 
    (SELECT 
        f.CODIGO AS cd,
        s.demanda_formacao,
        s.grande_grupo,
        s.subgrupo_principal,
        s.subgrupo,
        f.TITULO AS familia,
        s.ocupacao,
        s.industrial_uniepro
     FROM 
        compliance_cbo_familia f
     LEFT JOIN 
        (SELECT 
            s.cd AS codigo,
            p.demanda_formacao,
            p.grande_grupo,
            p.subgrupo_principal,
            s.TITULO AS subgrupo,
            p.familia,
            p.ocupacao,
            p.industrial_uniepro
         FROM 
            compliance_cbo_sub_grupo s
         LEFT OUTER JOIN 
            compliance_flat_sub_grupo_principal p ON s.cd_subgrupo_principal = p.codigo) AS s
     ON 
        f.cd_subgrupo = s.codigo) AS f
ON 
    o.cd_familia = f.cd

UNION ALL

SELECT 
    cd AS cd,
    demanda_formacao,
    grande_grupo AS dsc_grande_grupo,
    subgrupo_principal AS dsc_subgrupo_principal,
    subgrupo AS dsc_subgrupo,
    familia AS dsc_formacao,
    ocupacao AS dsc_ocupacao,
    'GRANDE GRUPO' AS dsc_nivel,
    industrial_uniepro AS dsc_industrial_uniepro
FROM 
    compliance_cbo_grande_grupo

UNION ALL

SELECT 
    codigo AS cd,
    demanda_formacao,
    grande_grupo AS dsc_grande_grupo,
    subgrupo_principal AS dsc_subgrupo_principal,
    subgrupo AS dsc_subgrupo,
    familia AS dsc_formacao,
    ocupacao AS dsc_ocupacao,
    'SUBGRUPO PRINCIPAL' AS dsc_nivel,
    industrial_uniepro AS dsc_industrial_uniepro
FROM 
    compliance_flat_sub_grupo_principal;
"""

In [9]:
compliance_flat_cbo_final = extrator_bronze.carregar_dados_delta(sql_query)

2024-09-26 19:29:27,939 - INFO - Aguarde... Executando Query (Delta)
2024-09-26 19:29:27,939 - INFO - 
SELECT 
    s.cd AS cd,
    p.demanda_formacao,
    p.grande_grupo AS dsc_grande_grupo,
    p.subgrupo_principal AS dsc_subgrupo_principal,
    s.TITULO AS dsc_subgrupo,
    p.familia AS dsc_formacao,
    p.ocupacao AS dsc_ocupacao,
    'SUBGRUPO' AS dsc_nivel,
    p.industrial_uniepro AS dsc_industrial_uniepro
FROM 
    compliance_cbo_sub_grupo s
LEFT OUTER JOIN 
    compliance_flat_sub_grupo_principal p ON s.cd_subgrupo_principal = p.codigo

UNION ALL

SELECT 
    f.CODIGO AS cd,
    s.demanda_formacao,
    s.grande_grupo AS dsc_grande_grupo,
    s.subgrupo_principal AS dsc_subgrupo_principal,
    s.subgrupo AS dsc_subgrupo,
    f.TITULO AS dsc_formacao,
    s.ocupacao AS dsc_ocupacao,
    'FAMILIA' AS dsc_nivel,
    s.industrial_uniepro AS dsc_industrial_uniepro
FROM 
    compliance_cbo_familia f
LEFT JOIN 
    (SELECT 
        s.cd AS codigo,
        p.demanda_formacao,
        p.

# Gravação no datalake

In [10]:
extrator_bronze.caminho_tabela_delta = 's3a://gold/Mte/FlatCBOFinal'

In [11]:
extrator_bronze.salvar_delta(compliance_flat_cbo_final, 'overwrite')

2024-09-26 19:29:28,096 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-26 19:29:47,640 - INFO - Dados persistidos com sucesso
2024-09-26 19:29:47,641 - INFO - s3a://gold/Mte/FlatCBOFinal
2024-09-26 19:29:47,641 - INFO - Aguarde... Realizando optimize
2024-09-26 19:29:49,297 - INFO - Optimize executado com sucesso.
2024-09-26 19:29:49,297 - INFO - Aguarde... Realizando vacuum
2024-09-26 19:30:01,134 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [12]:
extrator_bronze.parar_sessao()

2024-09-26 19:30:01,832 - INFO - Sessão Spark finalizada.
